In [ ]:
import polars as pl
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score

# 1. SETUP FOLDERS
os.makedirs("models", exist_ok=True)

def load_data(path):
    if os.path.exists(path):
        # Agar CSV mein separator ; hai to separator=";" add karo, wrna hata do
        return pl.read_csv(path) 
    else:
        print("data.CSV")
        return pl.DataFrame({
            "StudyHours": [2.5, 5.0, 8.0, 1.0, 4.0, 6.0, 3.0, 7.5, 9.0, 2.0],
            "Attendance": [70, 85, 95, 50, 75, 80, 65, 90, 98, 55],
            "Gender": ["M", "F", "M", "M", "F", "M", "F", "F", "M", "M"],
            "City": ["Delhi", "Mumbai", "Pune", "Mumbai", "Delhi", "Pune", "Mumbai", "Delhi", "Pune", "Delhi"],
            "Pass": [0, 1, 1, 0, 1, 1, 0, 1, 1, 0]
        })

df = load_data("data.csv")

df = df.with_columns([
    pl.col("StudyHours").fill_null(pl.col("StudyHours").median()),
    (pl.col("StudyHours") + pl.col("Attendance")).alias("Engagement")
])

le = LabelEncoder()
df = df.with_columns(pl.Series("Gender", le.fit_transform(df["Gender"].to_numpy())))
df = df.to_dummies(columns=["City"])

y = df["Pass"].to_numpy()
X = df.drop("Pass").to_numpy()

X = SelectKBest(score_func=f_classif, k=3).fit_transform(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
joblib.dump(scaler, "models/scaler.pkl")

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
joblib.dump(model, "models/model.pkl")

# 8. VERIFICATION
preds = model.predict(X_test)
print(f"Pipeline Execution Successful! Accuracy: {accuracy_score(y_test, preds):.2f}")
print(df.head())

data.CSV
Pipeline Execution Successful! Accuracy: 1.00
shape: (5, 8)
┌────────────┬────────────┬────────┬────────────┬─────────────┬───────────┬──────┬────────────┐
│ StudyHours ┆ Attendance ┆ Gender ┆ City_Delhi ┆ City_Mumbai ┆ City_Pune ┆ Pass ┆ Engagement │
│ ---        ┆ ---        ┆ ---    ┆ ---        ┆ ---         ┆ ---       ┆ ---  ┆ ---        │
│ f64        ┆ i64        ┆ i64    ┆ u8         ┆ u8          ┆ u8        ┆ i64  ┆ f64        │
╞════════════╪════════════╪════════╪════════════╪═════════════╪═══════════╪══════╪════════════╡
│ 2.5        ┆ 70         ┆ 1      ┆ 1          ┆ 0           ┆ 0         ┆ 0    ┆ 72.5       │
│ 5.0        ┆ 85         ┆ 0      ┆ 0          ┆ 1           ┆ 0         ┆ 1    ┆ 90.0       │
│ 8.0        ┆ 95         ┆ 1      ┆ 0          ┆ 0           ┆ 1         ┆ 1    ┆ 103.0      │
│ 1.0        ┆ 50         ┆ 1      ┆ 0          ┆ 1           ┆ 0         ┆ 0    ┆ 51.0       │
│ 4.0        ┆ 75         ┆ 0      ┆ 1          ┆ 0           ┆ 0  

In [ ]:
import polars as pl
import logging
import time
import os

# 1. SETUP LOGGING (Professional standard)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def run_production_pipeline(input_path: str, output_path: str):
    start_time = time.time()
    
    try:
        logger.info(f"Starting pipeline for {input_path}")
        
        # 2. LAZY LOADING WITH SCHEMA VALIDATION
        # Schema ensure karta hai ki data type sahi ho
        q = (
            pl.scan_csv(input_path)
            .with_columns([
                (pl.col("score") / 20 * 100).alias("normalized_score"),
                pl.col("city").fill_null("Unknown")
            ])
            # 3. ADVANCED WINDOW FUNCTION
            # Har student ka score uski City ke average se kitna alag hai
            .with_columns(
                (pl.col("normalized_score") - pl.col("normalized_score").mean().over("city"))
                .alias("performance_delta")
            )
            # 4. COMPLEX AGGREGATION
            .group_by("city")
            .agg([
                pl.col("normalized_score").mean().alias("city_avg"),
                pl.col("performance_delta").std().alias("performance_volatility"),
                pl.len().alias("student_count")
            ])
        )

        # 5. STREAMING SINK (Big Data Optimized)
        q.sink_parquet(output_path)
        
        duration = time.time() - start_time
        logger.info(f"Pipeline finished in {duration:.2f} seconds. Output: {output_path}")

    except Exception as e:
        logger.error(f"Pipeline crashed: {e}")

if __name__ == "__main__":
    os.makedirs("output", exist_ok=True)
    # Check if data exists, else handle gracefully
    if os.path.exists("data.csv"):
        run_production_pipeline("data.csv", "output/pro_metrics.parquet")
    else:
        logger.warning("data.csv not found. Pipeline skipped.")
        print(df.head())

2026-05-12 23:43:03,104 - WARNING - data.csv not found. Pipeline skipped.
